# Generation Eval Results — 2026-03-25 1:20AM

Re-run after increasing highlight `fragment_size` from 500 → 1500 and `number_of_fragments` from 1 → 3 in both the backend and eval pipeline.

## Changes from 12:30AM run

- `fragment_size`: 500 → 1500
- `number_of_fragments`: 1 → 3

## Key Findings

- **Faithfulness improved significantly** — overall up from 3.00 to ~4.00, with `rrf_elser` gaining +2.00 on gameplay questions specifically
- **Correctness also improved** — overall up from ~3.15 to ~3.65, `rrf_elser` now slightly edges `best_fields`
- **Source score increased** — from 1.10/1.30 to 1.40/1.70, confirming larger fragments surface more narrative content
- **Delta decreased** — from 1.30/1.10 to 1.10/0.80, meaning the model is adding less narrative on top of richer source material
- **`rrf_elser` flagged 2 questions vs 3 for `best_fields`** — hybrid search benefiting more from the larger fragment size
- Embellishment still concentrated in lore, particularly Sylvanas flagged under both strategies

In [1]:
import sys, json
from pathlib import Path
sys.path.append(str(Path('../..').resolve()))

import anthropic
from elasticsearch import Elasticsearch
from config import ANTHROPIC_API_KEY, ES_ENDPOINT, ES_API_KEY, ES_INDEX
from evals.generation.judge import score, score_narrative
from evals.generation.run_eval import search_with_snippets, generate

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
es     = Elasticsearch(ES_ENDPOINT, api_key=ES_API_KEY)

BENCHMARK          = Path('../../evals/generation/benchmark.json')
DIMENSIONS         = ['correctness', 'faithfulness', 'relevance']
COMPARE_STRATEGIES = ['best_fields', 'rrf_elser']

In [2]:
benchmark  = json.loads(BENCHMARK.read_text())
categories = sorted(set(b['category'] for b in benchmark))

print(f'Total questions: {len(benchmark)}')
for cat in categories:
    print(f'  {cat}: {sum(1 for b in benchmark if b["category"] == cat)}')

Total questions: 10
  gameplay: 2
  item: 2
  lore: 6


## Generation Eval Results

Loaded from `evals/generation/results.json`.

In [3]:
gen_results = json.loads(Path('../../evals/generation/results.json').read_text())

col = 14
print(f"{'Dimension':<15}", end='')
for name in COMPARE_STRATEGIES:
    print(f'  {name:>{col}}', end='')
print(f'  {"delta":>8}')
print('-' * (15 + (col + 2) * len(COMPARE_STRATEGIES) + 10))

for dim in DIMENSIONS:
    avgs  = {n: sum(r[dim] for r in gen_results[n]) / len(gen_results[n]) for n in COMPARE_STRATEGIES}
    delta = avgs[COMPARE_STRATEGIES[-1]] - avgs[COMPARE_STRATEGIES[0]]
    print(f'{dim:<15}', end='')
    for name in COMPARE_STRATEGIES:
        print(f'  {avgs[name]:>{col}.2f}', end='')
    print(f'  {delta:>+8.2f}')

Dimension           best_fields       rrf_elser     delta
---------------------------------------------------------
correctness                3.60            3.70     +0.10
faithfulness               3.90            4.10     +0.20
relevance                  4.60            4.60     +0.00


In [4]:
for category in categories:
    print(f'\n{category}')
    for dim in DIMENSIONS:
        avgs = []
        print(f'  {dim:<15}', end='')
        for name in COMPARE_STRATEGIES:
            cat_results = [r for r in gen_results[name] if r['category'] == category]
            avg = sum(r[dim] for r in cat_results) / len(cat_results)
            avgs.append(avg)
            print(f'  {avg:>12.2f}', end='')
        print(f'  {avgs[-1] - avgs[0]:>+8.2f}')


gameplay
  correctness              3.50          4.00     +0.50
  faithfulness             2.00          4.00     +2.00
  relevance                5.00          5.00     +0.00

item
  correctness              1.50          2.50     +1.00
  faithfulness             4.00          4.00     +0.00
  relevance                3.50          3.50     +0.00

lore
  correctness              4.33          4.00     -0.33
  faithfulness             4.50          4.17     -0.33
  relevance                4.83          4.83     +0.00


## Embellishment Eval Results

Loaded from `evals/generation/embellishment_results.json`.

In [5]:
emb_results = json.loads(Path('../../evals/generation/embellishment_results.json').read_text())

col = 14
print(f"{'':18}", end='')
for name in COMPARE_STRATEGIES:
    print(f'  {name:>{col}}', end='')
print()
print('-' * (18 + (col + 2) * len(COMPARE_STRATEGIES)))

for field in ['source_score', 'response_score', 'delta']:
    avgs = {n: sum(r[field] for r in emb_results[n]) / len(emb_results[n]) for n in COMPARE_STRATEGIES}
    print(f'{field:<18}', end='')
    for name in COMPARE_STRATEGIES:
        print(f'  {avgs[name]:>{col}.2f}', end='')
    print()

                       best_fields       rrf_elser
--------------------------------------------------
source_score                  1.40            1.70
response_score                2.50            2.50
delta                         1.10            0.80


In [6]:
for category in categories:
    print(f'\n{category}')
    for field in ['source_score', 'response_score', 'delta']:
        print(f'  {field:<18}', end='')
        for name in COMPARE_STRATEGIES:
            cat_results = [r for r in emb_results[name] if r['category'] == category]
            avg = sum(r[field] for r in cat_results) / len(cat_results)
            print(f'  {avg:>12.2f}', end='')
        print()


gameplay
  source_score              1.00          1.50
  response_score            2.00          2.00
  delta                     1.00          0.50

item
  source_score              1.50          1.50
  response_score            2.00          2.00
  delta                     0.50          0.50

lore
  source_score              1.50          1.83
  response_score            2.83          2.83
  delta                     1.33          1.00


In [7]:
for name in COMPARE_STRATEGIES:
    flagged = [r for r in emb_results[name] if r['delta'] >= 2]
    print(f'{name} — {len(flagged)} flagged')
    for r in flagged:
        print(f"  [{r['category']}] {r['question']}")
        print(f"    source={r['source_score']}  response={r['response_score']}  delta={r['delta']:+d}")
        print(f"    {r['response_reasoning']}")
    print()

best_fields — 3 flagged
  [lore] What is Frostmourne?
    source=1  response=3  delta=+2
    The text flows with some engaging prose and connective elements, but relies heavily on bolded topic sentences and maintains a somewhat structured, encyclopedic tone typical of lore summaries rather than fully immersive narrative storytelling.

  [lore] What is the Wrathgate?
    source=1  response=3  delta=+2
    The text uses flowing sentences and cause-and-effect to describe events, but is structured with headers and bolded key terms, and reads more as informed summary than immersive narrative prose.

  [lore] Who is Sylvanas Windrunner?
    source=2  response=4  delta=+2
    The text employs flowing, connected prose with clear cause-and-effect storytelling and a consistent voice throughout, though it maintains some structural elements through bolded topic sentences that slightly interrupt the pure narrative flow.

rrf_elser — 2 flagged
  [lore] Who is Sylvanas Windrunner?
    source=2  respo